# CocoAnalytica CNN Experiment Notebook

This notebook is the controlled experimentation workspace for the same four-class TensorFlow model used by CocoAnalytica.

**Classes:** Healthy, Yellowing, Coconut Scale Insect, and Rhinoceros Beetle.

It does not run inside the website. Every run creates a separate candidate model and records the test metrics. The active website model is changed only by the final, manually enabled promotion cell after the candidate passes the 80% overall and per-class F1 quality gate.

`Non-palms` remains excluded until its dataset has enough verified categories and images.


## Experiment Rules

1. Keep `balanced_dataset/test` untouched during model selection.
2. Change one experiment setting at a time and record the result.
3. Never improve a score by copying test images into training.
4. Use the saved evaluation JSON for the thesis results, not confidence values from individual uploads.
5. A candidate is not promoted unless overall test accuracy and every class F1 are at least 80%.


In [ ]:
from __future__ import annotations

import json
import csv
import random
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "Thesis AI Model").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

DATASET_DIR = PROJECT_ROOT / "Thesis AI Model" / "balanced_dataset"
OUTPUT_DIR = PROJECT_ROOT / "Thesis AI Model" / "model_outputs"
CANDIDATE_ROOT = OUTPUT_DIR / "candidates"
LIVE_MODEL_PATH = OUTPUT_DIR / "coconut_leaf_multilabel_cnn.keras"
LIVE_LABEL_CONFIG_PATH = OUTPUT_DIR / "label_config.json"

CLASS_NAMES = ["Healthy", "Yellowing", "Coconut_Scale_Insect", "Rhinoceros_Beetle"]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 20260910
RELEASE_THRESHOLD = 0.80

np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_DIR)
print("Live model:", LIVE_MODEL_PATH)


In [ ]:
manifest_path = DATASET_DIR / "manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run backend/scripts/prepare_balanced_cnn_dataset.py before using this notebook.")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("Candidate sources:", manifest["candidate_sources"])
print("Available after duplicate removal:", manifest["available_after_deduplication"])
print("Selected per class:", manifest["selected_per_class"])

for split in ("train", "validation", "test"):
    counts = {name: len(list((DATASET_DIR / split / name).glob("*"))) for name in CLASS_NAMES}
    print(f"{split}: {counts}")
    if len(set(counts.values())) != 1:
        raise ValueError(f"{split} is not balanced: {counts}")


In [ ]:
dataset_args = dict(
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

train = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "train", shuffle=True, seed=SEED, **dataset_args
)
validation = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "validation", shuffle=False, **dataset_args
)
test = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR / "test", shuffle=False, **dataset_args
)

AUTOTUNE = tf.data.AUTOTUNE
train = train.shuffle(512, seed=SEED, reshuffle_each_iteration=True).prefetch(AUTOTUNE)
validation = validation.prefetch(AUTOTUNE)
test = test.prefetch(AUTOTUNE)


## Choose One Controlled Experiment

Start with `head_256_finetune_40`. Only change one setting per run. Suggested follow-up variants are `finetune_80` and `focal_loss`; compare their saved JSON metrics, not training accuracy alone.


In [ ]:
EXPERIMENT = {
    "name": "head_256_finetune_40",
    "head_units": 256,
    "dropout": 0.30,
    "fine_tune_layers": 40,
    "head_epochs": 18,
    "fine_tune_epochs": 12,
    "head_learning_rate": 1e-3,
    "fine_tune_learning_rate": 1e-5,
    "loss": "cross_entropy",  # Change to "focal" only for a separate comparison run.
}

augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.06, seed=SEED),
        tf.keras.layers.RandomZoom(0.10, seed=SEED),
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name="field_photo_augmentation",
)

class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma: float = 1.5, name: str = "sparse_categorical_focal_loss"):
        super().__init__(name=name)
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0)
        selected = tf.gather(y_pred, y_true, axis=1, batch_dims=1)
        return -tf.pow(1.0 - selected, self.gamma) * tf.math.log(selected)


def build_model(config: dict[str, object]) -> tuple[tf.keras.Model, tf.keras.Model]:
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMAGE_SIZE, 3),
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = augmentation(inputs)
    x = tf.keras.applications.efficientnet.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = tf.keras.layers.BatchNormalization(name="head_batch_normalization")(x)
    x = tf.keras.layers.Dense(int(config["head_units"]), activation="relu", name="classification_features")(x)
    x = tf.keras.layers.Dropout(float(config["dropout"]), name="head_dropout")(x)
    outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="condition")(x)
    return tf.keras.Model(inputs, outputs, name="cocoanalytics_efficientnetb0"), base


def choose_loss(name: str):
    if name == "focal":
        return SparseCategoricalFocalLoss()
    return tf.keras.losses.SparseCategoricalCrossentropy()


def compile_model(model: tf.keras.Model, learning_rate: float, loss_name: str) -> None:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=choose_loss(loss_name),
        metrics=["accuracy"],
    )


In [ ]:
run_id = f"{EXPERIMENT['name']}_{datetime.now():%Y%m%d-%H%M%S}"
run_dir = CANDIDATE_ROOT / run_id
run_dir.mkdir(parents=True, exist_ok=True)

model, base_model = build_model(EXPERIMENT)
compile_model(model, float(EXPERIMENT["head_learning_rate"]), str(EXPERIMENT["loss"]))

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    tf.keras.callbacks.CSVLogger(run_dir / "head_training_log.csv"),
]

head_history = model.fit(
    train,
    validation_data=validation,
    epochs=int(EXPERIMENT["head_epochs"]),
    callbacks=callbacks,
)

base_model.trainable = True
fine_tune_at = max(len(base_model.layers) - int(EXPERIMENT["fine_tune_layers"]), 0)
for index, layer in enumerate(base_model.layers):
    layer.trainable = index >= fine_tune_at
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

compile_model(model, float(EXPERIMENT["fine_tune_learning_rate"]), str(EXPERIMENT["loss"]))
fine_tune_history = model.fit(
    train,
    validation_data=validation,
    epochs=int(EXPERIMENT["head_epochs"]) + int(EXPERIMENT["fine_tune_epochs"]),
    initial_epoch=len(head_history.history["loss"]),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        tf.keras.callbacks.CSVLogger(run_dir / "fine_tune_log.csv"),
    ],
)


In [ ]:
def calculate_metrics(model: tf.keras.Model, dataset: tf.data.Dataset) -> dict[str, object]:
    probabilities = model.predict(dataset, verbose=0)
    predicted = np.argmax(probabilities, axis=1)
    actual = np.concatenate([labels.numpy() for _, labels in dataset], axis=0)

    confusion = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)
    for truth, guess in zip(actual, predicted):
        confusion[int(truth), int(guess)] += 1

    per_class = {}
    for index, name in enumerate(CLASS_NAMES):
        true_positive = int(confusion[index, index])
        false_positive = int(confusion[:, index].sum() - true_positive)
        false_negative = int(confusion[index, :].sum() - true_positive)
        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(true_positive + false_negative, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        per_class[name] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1_score": float(f1),
            "support": int(confusion[index, :].sum()),
        }

    accuracy = float(np.mean(predicted == actual))
    return {
        "test_accuracy": accuracy,
        "confusion_matrix": confusion.tolist(),
        "per_class_metrics": per_class,
    }

metrics = calculate_metrics(model, test)
metrics["experiment"] = EXPERIMENT
metrics["dataset_split"] = "balanced_dataset/test; untouched during training"
metrics["release_threshold"] = RELEASE_THRESHOLD
metrics["passed_release_gate"] = (
    metrics["test_accuracy"] >= RELEASE_THRESHOLD
    and all(row["f1_score"] >= RELEASE_THRESHOLD for row in metrics["per_class_metrics"].values())
)

for name, row in metrics["per_class_metrics"].items():
    print(
        f"{name}: precision={row['precision']:.1%}, recall={row['recall']:.1%}, "
        f"f1={row['f1_score']:.1%}, support={row['support']}"
    )


In [ ]:
def collect_predictions(model, dataset):
    probabilities, labels = [], []
    for images, batch_labels in dataset:
        probabilities.append(model(images, training=False).numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(probabilities), np.concatenate(labels)


def test_time_augmented_predictions(model, dataset):
    probabilities, labels = [], []
    for images, batch_labels in dataset:
        original = model(images, training=False)
        flipped = model(tf.image.flip_left_right(images), training=False)
        probabilities.append(((original + flipped) / 2.0).numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(probabilities), np.concatenate(labels)


def accuracy_from_probabilities(probabilities, labels):
    return float(np.mean(np.argmax(probabilities, axis=1) == labels))


def fit_temperature(probabilities, labels):
    """Fit temperature on validation only; lower validation cross-entropy wins."""
    temperatures = np.linspace(0.5, 3.0, 51)
    logits = np.log(np.clip(probabilities, 1e-7, 1.0))
    losses = []
    for temperature in temperatures:
        scaled = logits / temperature
        scaled -= scaled.max(axis=1, keepdims=True)
        calibrated = np.exp(scaled)
        calibrated /= calibrated.sum(axis=1, keepdims=True)
        losses.append(-np.mean(np.log(np.clip(calibrated[np.arange(len(labels)), labels], 1e-7, 1.0))))
    return float(temperatures[int(np.argmin(losses))])

raw_test_probabilities, test_labels = collect_predictions(model, test)
tta_probabilities, _ = test_time_augmented_predictions(model, test)
validation_probabilities, validation_labels = collect_predictions(model, validation)
calibration_temperature = fit_temperature(validation_probabilities, validation_labels)
metrics["test_time_augmented_accuracy"] = accuracy_from_probabilities(tta_probabilities, test_labels)
metrics["calibration_temperature"] = calibration_temperature
print("Normal test accuracy:", f"{metrics['test_accuracy']:.1%}")
print("Test-time augmentation accuracy:", f"{metrics['test_time_augmented_accuracy']:.1%}")
print("Validation-only calibration temperature:", calibration_temperature)


In [ ]:
print("Overall test accuracy:", f"{metrics['test_accuracy']:.1%}")
print("Passed release gate:", metrics["passed_release_gate"])
print("Confusion matrix (rows=true, columns=predicted):")
for name, row in zip(CLASS_NAMES, metrics["confusion_matrix"]):
    print(name, row)

candidate_model_path = run_dir / "coconut_leaf_model.keras"
model.save(candidate_model_path)
label_config = {
    "class_names": CLASS_NAMES,
    "thresholds": {name: 0.5 for name in CLASS_NAMES},
    "uncertain_threshold": 0.5,
    "min_top_two_margin": 0.10,
    "temperature": calibration_temperature,
    "image_size": list(IMAGE_SIZE),
}
(run_dir / "label_config.json").write_text(json.dumps(label_config, indent=2), encoding="utf-8")
(run_dir / "evaluation.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

summary_path = CANDIDATE_ROOT / "experiment_results.csv"
summary_row = {
    "run_id": run_id,
    "test_accuracy": metrics["test_accuracy"],
    "passed_release_gate": metrics["passed_release_gate"],
    **{f"{name}_f1": row["f1_score"] for name, row in metrics["per_class_metrics"].items()},
}
existing_rows = []
if summary_path.exists():
    with summary_path.open("r", newline="", encoding="utf-8") as file:
        existing_rows = list(csv.DictReader(file))
all_rows = [*existing_rows, summary_row]
with summary_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_row))
    writer.writeheader()
    writer.writerows(all_rows)

print("Candidate saved to:", candidate_model_path)
print("Evaluation saved to:", run_dir / "evaluation.json")
print("Comparison table:", summary_path)


## Manual Promotion Only

Leave `PROMOTE_CANDIDATE = False` while experimenting. Set it to `True` only after reviewing the saved evaluation JSON and confirming that the candidate passed the quality gate. The cell refuses to copy a failing candidate into the model used by the website.


In [ ]:
PROMOTE_CANDIDATE = False

if PROMOTE_CANDIDATE:
    if not metrics["passed_release_gate"]:
        raise RuntimeError("This candidate did not pass the 80% overall and per-class F1 release gate.")
    shutil.copy2(candidate_model_path, LIVE_MODEL_PATH)
    LIVE_LABEL_CONFIG_PATH.write_text(json.dumps(label_config, indent=2), encoding="utf-8")
    (OUTPUT_DIR / "evaluation.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print("Candidate promoted to the website model:", LIVE_MODEL_PATH)
else:
    print("Promotion disabled. The live website model was not changed.")
